# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# --- Install (if needed) ---
!pip install huggingface_hub pandas pyarrow -q

# --- Auth via Colab Secrets ---
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# --- Load the March 2026 partition of fact_content_daily_performance ---
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())

(9841378, 31)
2026-03-01 2026-03-31


In [5]:
dim_clients = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet")
dim_content = pd.read_parquet("hf://datasets/FlyRank/internship-warehouse/dim_content.parquet")

print(dim_clients.shape, dim_content.shape)
print(dim_clients.columns.tolist())
print(dim_content.columns.tolist())

(104, 9) (519606, 26)
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [8]:
print(df_march.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [13]:
import pandas as pd
import numpy as np

df = df_march.copy()

# --- Join dimension tables ---
df = df.merge(dim_content[['content_hash_id', 'content_type', 'word_count', 'content_created_date']],
              on='content_hash_id', how='left')

# --- Filter to rows where GSC data is available ---
df = df[df['gsc_data_available'] == True]

# --- Filter out very low-impression rows (ML-02 cutoff) ---
df = df[df['gsc_impressions'] >= 500]

# --- BUILD THE LABEL FIRST ---
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']

df['engagement_rate'] = np.where(
    df['ga4_data_available'] == True,
    df['ga4_engaged_sessions'] / df['ga4_sessions'].replace(0, np.nan),
    np.nan
)

df['impr_rank'] = df.groupby('client_hash_id')['gsc_impressions'].rank(pct=True)
df['engagement_rank'] = df.groupby('client_hash_id')['ctr'].rank(pct=True)

df['is_opportunity'] = (
    (df['impr_rank'] >= 0.67) &
    (df['engagement_rank'] <= 0.33)
).astype(int)

print("Label balance:", df['is_opportunity'].value_counts(normalize=True))

# --- NOW build features ---
# BANNED (label-derived): gsc_impressions, gsc_clicks, ctr, ga4_engaged_sessions, ga4_sessions,
#                          engagement_rate, impr_rank, engagement_rank

df = pd.get_dummies(df, columns=['content_type'], prefix='ctype')

df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count_filled'] = df['word_count'].fillna(0)

df['content_age_days'] = (
    pd.to_datetime(df['report_date']) - pd.to_datetime(df['content_created_date'])
).dt.days

df['has_position_data'] = (df['gsc_avg_position'] != 0).astype(int)
df['gsc_avg_position_filled'] = df['gsc_avg_position'].replace(0, np.nan).fillna(df['gsc_avg_position'].median())

# --- Traffic-source mix, with GA4-availability flag (avoids "0 vs no data" confusion) ---
df['has_ga4_access'] = (df['ga4_data_available'] == True).astype(int)

df['pct_sessions_organic'] = df['sessions_organic'] / df['ga4_sessions'].replace(0, np.nan)
df['pct_sessions_ai'] = df['sessions_ai'] / df['ga4_sessions'].replace(0, np.nan)

df['pct_sessions_organic'] = df['pct_sessions_organic'].fillna(0)
df['pct_sessions_ai'] = df['pct_sessions_ai'].fillna(0)

feature_cols = [
    'has_word_count', 'word_count_filled',
    'content_age_days',
    'has_position_data', 'gsc_avg_position_filled',
    'has_ga4_access', 'pct_sessions_organic', 'pct_sessions_ai',
] + [c for c in df.columns if c.startswith('ctype_')]

X = df[feature_cols]
y = df['is_opportunity']

print(X.shape, y.shape)
print(X.isna().sum())

Label balance: is_opportunity
0    0.889385
1    0.110615
Name: proportion, dtype: float64
(101451, 11) (101451,)
has_word_count              0
word_count_filled           0
content_age_days            0
has_position_data           0
gsc_avg_position_filled     0
has_ga4_access              0
pct_sessions_organic        0
pct_sessions_ai             0
ctype_comparison article    0
ctype_feedly article        0
ctype_keyword article       0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Section 2: Feature Notes

**has_word_count**
Meaning: Flag — 1 if word_count was present in dim_content, 0 if it was null.
Missing: N/A — this is the missingness indicator itself.
Available before prediction? Yes — set at content creation, static.

**word_count_filled**
Meaning: Word count of the content piece; 0 where originally missing (pair with has_word_count to tell real 0 from missing).
Missing: Filled with 0.
Available before prediction? Yes — known at content creation, before any report_date.

**content_age_days**
Meaning: Days between content_created_date and the report_date being scored.
Missing: None — both source dates always populated.
Available before prediction? Yes — by definition, always known at prediction time.

**has_position_data**
Meaning: Flag — 1 if gsc_avg_position is a real value, 0 if it was 0 (warehouse convention for "no data").
Missing: N/A — this is the missingness indicator itself.
Available before prediction? Yes — reflects trailing GSC data up to report_date, not the future.

**gsc_avg_position_filled**
Meaning: Average search ranking position; 0-values (no-data) replaced with dataset median so the model isn't fed a false "rank 0."
Missing: Filled with column median.
Available before prediction? Yes — same as above.

**has_ga4_access**
Meaning: Flag — 1 if this client had GA4 tracking confirmed active on this row, 0 if not confirmed/unknown.
Missing: N/A — this is the missingness indicator itself.
Available before prediction? Yes — a client-level setup fact, independent of any single day's outcome.

**pct_sessions_organic**
Meaning: Share of GA4 sessions from organic search traffic; 0 where GA4 unavailable.
Missing: Filled with 0, paired with has_ga4_access flag.
Available before prediction? Yes — reflects trailing traffic mix, not the label window outcome.

**pct_sessions_ai**
Meaning: Share of GA4 sessions from AI-assistant referral traffic (ChatGPT, Perplexity, etc.); 0 where GA4 unavailable.
Missing: Filled with 0, paired with has_ga4_access flag.
Available before prediction? Yes — same reasoning as above.

**ctype_comparison article**
Meaning: One-hot — 1 if content is a "comparison article."
Missing: None — categorical, always populated.
Available before prediction? Yes — static content attribute set at creation.

**ctype_feedly article**
Meaning: One-hot — 1 if content is a "feedly article."
Missing: None.
Available before prediction? Yes — same as above.

**ctype_keyword article**
Meaning: One-hot — 1 if content is a "keyword article."
Missing: None.
Available before prediction? Yes — same as above.

**Caveat:** pct_sessions_organic and pct_sessions_ai are trailing/aggregate shares over whatever window the GA4 columns represent in this snapshot — not explicitly re-verified to only reflect time strictly before report_date within a day. This is a candidate leakage risk to test formally in Section 3.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Section 3: The Leakage Hunt

**Attack 1 — raw ctr:** Injected `gsc_clicks / gsc_impressions` (raw CTR) as a feature.
Result: AUC moved only 0.637 → 0.670 (+0.033). Small jump — raw CTR alone doesn't
fully expose the label logic, because the label depends on CTR's rank *within each
client*, not its raw value.

**Attack 2 — within-client rank versions:** Injected the exact rank features
(`ctr` percentile rank and `impressions` percentile rank, grouped by client) that
were used to construct `is_opportunity`. Result: AUC collapsed toward ceiling,
0.637 → 0.992 (+0.355). This confirms the test harness correctly detects leakage
when the leaky feature matches the label's actual construction mechanism.

**Conclusion:** `gsc_impressions`, `gsc_clicks`, `ctr`, and especially their
within-client percentile-rank versions are all banned from the feature set —
confirmed empirically, not just by inspection. The honest feature set (AUC 0.637,
base rate 0.889/0.111) shows a real, modest, non-leaked signal.

**Split integrity:** All AUC scores use GroupKFold split by `client_hash_id`,
so no client's content appears in both train and test — this is a grouped,
not random, evaluation, per the leakage-hunting checklist.

In [16]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

# --- Base rate first — every score gets compared to this ---
base_rate = y.mean()
print(f"Base rate (naive 'always predict majority class'): {1 - base_rate:.3f}")
print(f"Base rate (positive class share): {base_rate:.3f}")

# --- Scale numeric features (fixes convergence warning) ---
scaler = StandardScaler()
numeric_cols = ['word_count_filled', 'content_age_days', 'gsc_avg_position_filled',
                 'pct_sessions_organic', 'pct_sessions_ai']

X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_scaled[numeric_cols])

# --- Grouped split by client (not random) — per leakage skill ---
groups = df['client_hash_id']
gkf = GroupKFold(n_splits=5)

def run_cv(X_data, y_data, groups_data, label=""):
    scores = []
    for train_idx, test_idx in gkf.split(X_data, y_data, groups_data):
        X_train, X_test = X_data.iloc[train_idx], X_data.iloc[test_idx]
        y_train, y_test = y_data.iloc[train_idx], y_data.iloc[test_idx]
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        preds = model.predict_proba(X_test)[:, 1]
        scores.append(roc_auc_score(y_test, preds))
    print(f"{label} — grouped CV AUC: {np.mean(scores):.3f} (+/- {np.std(scores):.3f})")
    return np.mean(scores)

# --- TEST 1: honest features only (what we built in Section 1) ---
honest_auc = run_cv(X_scaled, y, groups, label="Honest features")

# # --- TEST 2 (revised): inject something closer to what actually built the label ---
# The label used within-client RANKS of both ctr and impressions — so a proper leak
# needs to expose that same relative information, not just raw ctr.
df['LEAKY_ctr_rank'] = df.groupby('client_hash_id')['ctr'].rank(pct=True)
df['LEAKY_impr_rank'] = df.groupby('client_hash_id')['gsc_impressions'].rank(pct=True)

X_leaky2 = X_scaled.copy()
X_leaky2['LEAKY_ctr_rank'] = df['LEAKY_ctr_rank']
X_leaky2['LEAKY_impr_rank'] = df['LEAKY_impr_rank']

leaky_auc2 = run_cv(X_leaky2, y, groups, label="With deliberate leak (rank versions)")

print(f"\nHonest AUC: {honest_auc:.3f} | Leaky (raw ctr) AUC: {leaky_auc:.3f} | Leaky (rank) AUC: {leaky_auc2:.3f}")

Base rate (naive 'always predict majority class'): 0.889
Base rate (positive class share): 0.111
Honest features — grouped CV AUC: 0.637 (+/- 0.034)
With deliberate leak (rank versions) — grouped CV AUC: 0.992 (+/- 0.003)

Honest AUC: 0.637 | Leaky (raw ctr) AUC: 0.670 | Leaky (rank) AUC: 0.992


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Section 4: What I Excluded and Why

**gsc_impressions** — Directly used to build the label (impression-rank half of is_opportunity). Using it as a feature would let the model see the answer.

**gsc_clicks** — Used with gsc_impressions to compute ctr, which built the label. Label-derived.

**ctr (gsc_clicks / gsc_impressions)** — The core engagement signal the label is built from. Confirmed empirically in Section 3 (self-test jump).

**impr_rank (within-client percentile rank of impressions)** — One half of the literal label formula. Confirmed as a severe leak in Section 3 (AUC jumped to 0.992 when included).

**engagement_rank (within-client percentile rank of ctr)** — The other half of the literal label formula. Same leakage confirmation as above.

**ga4_sessions** — Denominator used to build engagement_rate, a candidate label signal. Excluded to keep the label definition consistent and the feature set clean of anything touching the engagement calculation.

**ga4_engaged_sessions** — Numerator used to build engagement_rate. Same reasoning as ga4_sessions.

**engagement_rate** — Not used in the final label formula (I used GSC ctr instead), but built from banned columns above, so excluded as a precaution against accidental correlation with the label.

**client_hash_id** — Pseudonymized ID. Per the flyrank-data skill, IDs exist only for joining/grouping (used correctly here for the GroupKFold split), never as a model input — a model shouldn't learn "this specific client code = opportunity."

**content_hash_id** — Same reasoning as client_hash_id: join/grouping key only, never a feature.

**report_date** — Used to compute content_age_days (a legitimate feature), but the raw date itself is excluded — a specific calendar date isn't a generalizable pattern the model should learn from directly, and could encode date-specific noise from the March-only snapshot.

**gsc_sum_position** — Redundant with gsc_avg_position (which I did use, filled). Including both would double-count the same ranking signal.

**LEAKY_ctr, LEAKY_ctr_rank, LEAKY_impr_rank** — These were deliberately injected only for the Section 3 self-test, to prove the leakage-testing harness actually detects leakage when it's present. Never intended as real features; dropped from the final X.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.